In [1]:
!git config --global user.name "31Pritika"
!git config --global user.email "pritikaagarwal0731@gmail"


In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ---------------------------------------------------------------
# 1. Load the raw files
# ---------------------------------------------------------------
print("Loading files...")

# FULL FILE is 26M rows across ~270K users. Randomly sampling individual ROWS
# (what we tried before) spreads thin across everyone and destroys per-user
# signal — most users end up with only 1-9 ratings, too little for SVD to learn
# anything. Instead: sample a subset of USERS first, then keep ALL of their
# ratings. Fewer total users, but each one has a real, dense rating history —
# much closer to how ratings_small.csv behaved.
movies = pd.read_csv("/content/drive/MyDrive/moviesdataset/movies_metadata.csv", low_memory=False)

N_USERS_TO_SAMPLE = 30000  # tune up/down based on memory — this keeps full histories

# First pass: just read userId to see who exists, without loading everything
user_id_col = pd.read_csv("/content/drive/MyDrive/moviesdataset/ratings.csv", usecols=["userId"], dtype={"userId": np.int32})
all_user_ids = user_id_col["userId"].unique()
print(f"Total users in full file: {len(all_user_ids)}")

rng = np.random.default_rng(42)
sampled_user_ids = set(rng.choice(all_user_ids, size=min(N_USERS_TO_SAMPLE, len(all_user_ids)), replace=False))

# Second pass: read the file in chunks, keep only rows belonging to sampled users
kept_chunks = []
for chunk in pd.read_csv(
    "/content/drive/MyDrive/moviesdataset/ratings.csv",
    dtype={"userId": np.int32, "movieId": np.int32, "rating": np.float32},
    usecols=["userId", "movieId", "rating"],
    chunksize=500_000,
):
    kept_chunks.append(chunk[chunk["userId"].isin(sampled_user_ids)])

ratings = pd.concat(kept_chunks, ignore_index=True)
print(f"Loaded {len(ratings)} ratings from {ratings['userId'].nunique()} sampled users "
      f"(avg {len(ratings)/ratings['userId'].nunique():.1f} ratings/user — full histories kept)")

print(f"movies_metadata: {movies.shape}")
print(f"ratings: {ratings.shape}")


Loading files...
Total users in full file: 270896
Loaded 2886845 ratings from 30000 sampled users (avg 96.2 ratings/user — full histories kept)
movies_metadata: (45466, 24)
ratings: (2886845, 3)


In [5]:
# ---------------------------------------------------------------
# 2. Clean movies_metadata
# ---------------------------------------------------------------
movies = movies[movies["id"].astype(str).str.isnumeric()]
movies["id"] = movies["id"].astype(int)

movies_clean = movies[["id", "title", "genres", "overview", "vote_average", "vote_count"]].copy()
movies_clean = movies_clean.rename(columns={"id": "movieId_tmdb"})


In [6]:
# ---------------------------------------------------------------
# 3. Bridge MovieLens movieId -> TMDB id via links.csv
# ---------------------------------------------------------------
try:
    links = pd.read_csv("/content/drive/MyDrive/moviesdataset/links.csv", dtype={"movieId": np.int32})
    links = links.dropna(subset=["tmdbId"])
    links["tmdbId"] = links["tmdbId"].astype(np.int32)
    ratings = ratings.merge(links[["movieId", "tmdbId"]], on="movieId", how="inner")
    ratings = ratings.rename(columns={"tmdbId": "movieId_tmdb"})
    print("Linked ratings to TMDB ids via links.csv")
except FileNotFoundError:
    print("links.csv not found — skipping TMDB id bridge for now")
    ratings["movieId_tmdb"] = ratings["movieId"]


Linked ratings to TMDB ids via links.csv


In [7]:
# ---------------------------------------------------------------
# 4. Build a sparse user-item matrix
# ---------------------------------------------------------------
print("\nBuilding sparse user-item matrix...")

user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId_tmdb"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}
idx_to_user = {v: k for k, v in user_to_idx.items()}

row_idx = ratings["userId"].map(user_to_idx).values
col_idx = ratings["movieId_tmdb"].map(movie_to_idx).values
data = ratings["rating"].values

sparse_matrix = csr_matrix((data, (row_idx, col_idx)), shape=(len(user_ids), len(movie_ids)))

print(f"Sparse matrix shape: {sparse_matrix.shape}")
print(f"Non-zero entries: {sparse_matrix.nnz} "
      f"({100 * sparse_matrix.nnz / (sparse_matrix.shape[0]*sparse_matrix.shape[1]):.4f}% filled)")



Building sparse user-item matrix...
Sparse matrix shape: (30000, 31860)
Non-zero entries: 2885342 (0.3019% filled)


In [8]:
# ---------------------------------------------------------------
# 5. Sanity check
# ---------------------------------------------------------------
sample_user = ratings["userId"].iloc[0]
sample_ratings = ratings[ratings["userId"] == sample_user].merge(movies_clean, on="movieId_tmdb", how="left")
print(f"\nSample: user {sample_user}'s rated movies")
print(sample_ratings[["title", "rating"]].head(10))


Sample: user 1's rated movies
                      title  rating
0                Braveheart     1.0
1    The Basketball Diaries     4.5
2             The Godfather     5.0
3    The Godfather: Part II     5.0
4        Dead Poets Society     5.0
5        The Breakfast Club     4.0
6           The Sixth Sense     4.5
7  Ferris Bueller's Day Off     5.0
8                Fight Club     4.0
9                   Memento     4.0


In [9]:
# ---------------------------------------------------------------
# 6. SVD — compress users into taste patterns
# ---------------------------------------------------------------
print("\nRunning SVD to compress users into taste patterns...")
n_components = 20
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(sparse_matrix)

print(f"user_factors shape: {user_factors.shape}")
print(f"Variance explained: {svd.explained_variance_ratio_.sum():.2%}")



Running SVD to compress users into taste patterns...
user_factors shape: (30000, 20)
Variance explained: 31.50%


In [10]:
# ---------------------------------------------------------------
# 7. Similarity — the match score
# ---------------------------------------------------------------
print("\nComputing similarity for one sample user (not all-vs-all — that would be a "
      "users x users matrix, way too much memory at this scale)...")

# Compute similarity between ONE user and everyone else, not the full N x N grid.
# This is also how a real system would do it in production: on-demand, per user,
# not a giant precomputed matrix for the whole user base.
sample_idx = 0
sample_user_vector = user_factors[sample_idx:sample_idx+1]  # shape (1, n_components)
similarities_to_sample = cosine_similarity(sample_user_vector, user_factors)[0]  # shape (n_users,)

similarities_to_sample[sample_idx] = -1  # exclude comparing the user to themselves
most_similar_idx = similarities_to_sample.argmax()
most_similar_userId = idx_to_user[most_similar_idx]

print(f"\nMost similar user to userId {idx_to_user[sample_idx]}: userId {most_similar_userId} "
      f"(similarity: {similarities_to_sample[most_similar_idx]:.3f})")

similar_user_ratings = ratings[ratings["userId"] == most_similar_userId].merge(
    movies_clean, on="movieId_tmdb", how="left"
)
print(f"\nWhat userId {most_similar_userId} rated:")
print(similar_user_ratings[["title", "rating"]].head(10))



Computing similarity for one sample user (not all-vs-all — that would be a users x users matrix, way too much memory at this scale)...

Most similar user to userId 1: userId 263999 (similarity: 0.931)

What userId 263999 rated:
                            title  rating
0                         Jumanji     3.5
1  Ace Ventura: When Nature Calls     2.0
2                           Se7en     4.0
3                     Taxi Driver     4.5
4                       Star Wars     3.5
5                    Pulp Fiction     4.0
6                Schindler's List     4.5
7                   The Godfather     5.0
8  The Good, the Bad and the Ugly     4.5
9                      GoodFellas     4.5


In [11]:
# ---------------------------------------------------------------
# 8. Evaluate: precision@k
# ---------------------------------------------------------------
print("\nEvaluating with precision@k...")

LIKE_THRESHOLD = 4.0
K = 10

train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

train_row_idx = train_ratings["userId"].map(user_to_idx).values
train_col_idx = train_ratings["movieId_tmdb"].map(movie_to_idx).values
train_data = train_ratings["rating"].values

train_matrix = csr_matrix((train_data, (train_row_idx, train_col_idx)),
                           shape=(len(user_ids), len(movie_ids)))

svd_eval = TruncatedSVD(n_components=n_components, random_state=42)
train_user_factors = svd_eval.fit_transform(train_matrix)

# NOTE: we do NOT compute `train_user_factors @ svd_eval.components_` for ALL users
# at once — that builds a dense (users x movies) matrix, same memory problem as
# Block 7's similarity matrix. Instead, predict scores one user at a time, inside
# the loop below, only for the users we're actually evaluating.

precisions = []
test_users = test_ratings["userId"].unique()

# At full scale evaluating every single user is slow during development iteration.
# Sample a subset for fast feedback; rerun on the full test_users list once for
# your final reported number.
EVAL_SAMPLE_SIZE = 5000
if len(test_users) > EVAL_SAMPLE_SIZE:
    rng = np.random.default_rng(42)
    test_users = rng.choice(test_users, size=EVAL_SAMPLE_SIZE, replace=False)
    print(f"Sampling {EVAL_SAMPLE_SIZE} of {len(test_ratings['userId'].unique())} test users for speed "
          f"(rerun with the full list for your final report number)")

for u in test_users:
    if u not in user_to_idx:
        continue
    u_idx = user_to_idx[u]

    already_rated_train = set(train_ratings[train_ratings["userId"] == u]["movieId_tmdb"])
    already_rated_idx = {movie_to_idx[m] for m in already_rated_train if m in movie_to_idx}

    # Predict scores for THIS user only — a single row (n_components,) @ (n_components, n_movies)
    # = (n_movies,) vector. Tiny, compared to doing this for every user at once.
    user_scores = train_user_factors[u_idx] @ svd_eval.components_
    for idx in already_rated_idx:
        user_scores[idx] = -np.inf
    top_k_idx = np.argsort(user_scores)[::-1][:K]
    top_k_movie_ids = {movie_ids[i] for i in top_k_idx}

    actually_liked_test = set(
        test_ratings[(test_ratings["userId"] == u) & (test_ratings["rating"] >= LIKE_THRESHOLD)]["movieId_tmdb"]
    )

    if len(actually_liked_test) == 0:
        continue

    hits = len(top_k_movie_ids & actually_liked_test)
    precisions.append(hits / K)

mean_precision = np.mean(precisions)
print(f"\nEvaluated on {len(precisions)} users")
print(f"Mean precision@{K}: {mean_precision:.4f}")
print(f"(i.e. on average, {mean_precision*K:.2f} out of every {K} recommended movies "
      f"were ones the user actually rated >= {LIKE_THRESHOLD} in held-out data)")



Evaluating with precision@k...
Sampling 5000 of 27993 test users for speed (rerun with the full list for your final report number)

Evaluated on 4469 users
Mean precision@10: 0.1789
(i.e. on average, 1.79 out of every 10 recommended movies were ones the user actually rated >= 4.0 in held-out data)


In [12]:
# ---------------------------------------------------------------
# 9. K-Means — group users into taste archetypes
# ---------------------------------------------------------------
print("\nClustering users into taste archetypes...")

N_CLUSTERS = 8
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(user_factors)

print(f"Cluster sizes: {np.bincount(cluster_labels)}")

sample_cluster = 0
users_in_cluster = [idx_to_user[i] for i in range(len(cluster_labels)) if cluster_labels[i] == sample_cluster]
print(f"\nCluster {sample_cluster} has {len(users_in_cluster)} users. "
      f"Sample user: {users_in_cluster[0] if users_in_cluster else 'none'}")

if users_in_cluster:
    sample_cluster_user = users_in_cluster[0]
    top_rated = ratings[(ratings["userId"] == sample_cluster_user) & (ratings["rating"] >= 4.0)] \
        .merge(movies_clean, on="movieId_tmdb", how="left")
    print(f"Top-rated movies for a user in cluster {sample_cluster}:")
    print(top_rated[["title", "rating"]].head(10))



Clustering users into taste archetypes...
Cluster sizes: [18440   429  1253   260  2632  1776  4292   918]

Cluster 0 has 18440 users. Sample user: 1
Top-rated movies for a user in cluster 0:
                      title  rating
0    The Basketball Diaries     4.5
1             The Godfather     5.0
2    The Godfather: Part II     5.0
3        Dead Poets Society     5.0
4        The Breakfast Club     4.0
5           The Sixth Sense     4.5
6  Ferris Bueller's Day Off     5.0
7                Fight Club     4.0
8                   Memento     4.0
9              Donnie Darko     5.0


In [13]:
# ---------------------------------------------------------------
# 10. DIAGNOSTIC — how bad is the missing-metadata problem?
# ---------------------------------------------------------------
print("\n--- Diagnostic: missing metadata ---")

ratings_with_titles = ratings.merge(movies_clean, on="movieId_tmdb", how="left")
missing_pct = ratings_with_titles["title"].isna().mean() * 100
print(f"% of ratings with NO matching movie title: {missing_pct:.2f}%")

# Check WHY: are these tmdbIds just not present in movies_metadata.csv at all?
missing_ids = ratings_with_titles[ratings_with_titles["title"].isna()]["movieId_tmdb"].unique()
ids_in_movies_file = set(movies_clean["movieId_tmdb"])
truly_absent = [m for m in missing_ids if m not in ids_in_movies_file]
print(f"Of {len(missing_ids)} missing movie ids, {len(truly_absent)} don't exist in "
      f"movies_metadata.csv at all (the rest may have title as literally blank/NaN in the source file)")



--- Diagnostic: missing metadata ---
% of ratings with NO matching movie title: 0.12%
Of 115 missing movie ids, 114 don't exist in movies_metadata.csv at all (the rest may have title as literally blank/NaN in the source file)


In [14]:
# ---------------------------------------------------------------
# 11. DIAGNOSTIC — how many ratings does each user have?
# ---------------------------------------------------------------
print("\n--- Diagnostic: ratings per user ---")
ratings_per_user = ratings.groupby("userId").size()
print(ratings_per_user.describe())
print(f"\nUsers with fewer than 10 ratings: {(ratings_per_user < 10).sum()} "
      f"out of {len(ratings_per_user)} total users")



--- Diagnostic: ratings per user ---
count    30000.000000
mean        96.178733
std        229.198245
min          1.000000
25%         15.000000
50%         30.000000
75%         92.000000
max      18275.000000
dtype: float64

Users with fewer than 10 ratings: 4128 out of 30000 total users


In [15]:
# ---------------------------------------------------------------
# 12. FIX — drop ratings with missing metadata, filter low-activity users,
#     then redo SVD + clustering on the cleaner data
# ---------------------------------------------------------------
print("\n--- Rebuilding on cleaned data ---")

MIN_RATINGS_PER_USER = 10

# Drop ratings that have no matching movie metadata
ratings_clean = ratings_with_titles[ratings_with_titles["title"].notna()][
    ["userId", "movieId_tmdb", "rating"]
].copy()

# Keep only users with enough ratings to produce a stable taste fingerprint
active_users = ratings_per_user[ratings_per_user >= MIN_RATINGS_PER_USER].index
ratings_clean = ratings_clean[ratings_clean["userId"].isin(active_users)]

print(f"Ratings after cleaning: {len(ratings_clean)} (was {len(ratings)})")
print(f"Users after cleaning: {ratings_clean['userId'].nunique()} (was {len(user_ids)})")

# Rebuild id mappings on the cleaned data
user_ids_c = ratings_clean["userId"].unique()
movie_ids_c = ratings_clean["movieId_tmdb"].unique()
user_to_idx_c = {u: i for i, u in enumerate(user_ids_c)}
movie_to_idx_c = {m: i for i, m in enumerate(movie_ids_c)}
idx_to_user_c = {v: k for k, v in user_to_idx_c.items()}

row_idx_c = ratings_clean["userId"].map(user_to_idx_c).values
col_idx_c = ratings_clean["movieId_tmdb"].map(movie_to_idx_c).values
data_c = ratings_clean["rating"].values

sparse_matrix_c = csr_matrix((data_c, (row_idx_c, col_idx_c)),
                              shape=(len(user_ids_c), len(movie_ids_c)))

svd_c = TruncatedSVD(n_components=n_components, random_state=42)
user_factors_c = svd_c.fit_transform(sparse_matrix_c)

# Try a smaller number of clusters on the cleaned data
N_CLUSTERS_RETRY = 5
kmeans_c = KMeans(n_clusters=N_CLUSTERS_RETRY, random_state=42, n_init=10)
cluster_labels_c = kmeans_c.fit_predict(user_factors_c)

print(f"\nCluster sizes (cleaned data, {N_CLUSTERS_RETRY} clusters): {np.bincount(cluster_labels_c)}")

# Sanity check on the new cluster 0
sample_cluster_c = 0
users_in_cluster_c = [idx_to_user_c[i] for i in range(len(cluster_labels_c)) if cluster_labels_c[i] == sample_cluster_c]
print(f"\nCluster {sample_cluster_c} has {len(users_in_cluster_c)} users. "
      f"Sample user: {users_in_cluster_c[0] if users_in_cluster_c else 'none'}")

if users_in_cluster_c:
    sample_user_c = users_in_cluster_c[0]
    top_rated_c = ratings_clean[(ratings_clean["userId"] == sample_user_c) & (ratings_clean["rating"] >= 4.0)] \
        .merge(movies_clean, on="movieId_tmdb", how="left")
    print(f"Top-rated movies for a user in cluster {sample_cluster_c} (cleaned data):")
    print(top_rated_c[["title", "rating"]].head(10))



--- Rebuilding on cleaned data ---
Ratings after cleaning: 2863142 (was 2885362)
Users after cleaning: 25872 (was 30000)

Cluster sizes (cleaned data, 5 clusters): [ 3221  2122 17082   739  2708]

Cluster 0 has 3221 users. Sample user: 11
Top-rated movies for a user in cluster 0 (cleaned data):
                                 title  rating
0                                 Babe     4.0
1                         Pulp Fiction     4.0
2             The Shawshank Redemption     4.5
3                         Forrest Gump     4.0
4                       Mrs. Doubtfire     4.0
5                     Schindler's List     4.5
6             The Silence of the Lambs     4.0
7  Willy Wonka & the Chocolate Factory     4.0
8                              Titanic     4.0
9         There's Something About Mary     4.0


In [16]:
# ---------------------------------------------------------------
# 13. Elbow method — pick cluster count based on evidence, not a guess
# ---------------------------------------------------------------
print("\n--- Elbow method: testing cluster counts 2-10 ---")

inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(user_factors_c)
    inertias.append(km.inertia_)
    print(f"k={k}: inertia={km.inertia_:.2f}")

# Compute how much inertia DROPS with each additional cluster —
# a big drop means that extra cluster is genuinely useful,
# a tiny drop means it's not adding much structure.
print("\nInertia drop per step (bigger drop = more useful cluster):")
for i in range(1, len(inertias)):
    drop = inertias[i-1] - inertias[i]
    print(f"  {k_range[i-1]} -> {k_range[i]} clusters: drop = {drop:.2f}")

print("\nLook for where the drop-off flattens out (the 'elbow') — "
      "that's a defensible choice for final cluster count. "
      "Pick that k, rerun KMeans once more with it, and move on to FastAPI integration.")



--- Elbow method: testing cluster counts 2-10 ---
k=2: inertia=8090125.00
k=3: inertia=7179180.50
k=4: inertia=6597405.50
k=5: inertia=6067586.00
k=6: inertia=5717901.50
k=7: inertia=5461547.50
k=8: inertia=5244055.50
k=9: inertia=5063729.00
k=10: inertia=4917183.50

Inertia drop per step (bigger drop = more useful cluster):
  2 -> 3 clusters: drop = 910944.50
  3 -> 4 clusters: drop = 581775.00
  4 -> 5 clusters: drop = 529819.50
  5 -> 6 clusters: drop = 349684.50
  6 -> 7 clusters: drop = 256354.00
  7 -> 8 clusters: drop = 217492.00
  8 -> 9 clusters: drop = 180326.50
  9 -> 10 clusters: drop = 146545.50

Look for where the drop-off flattens out (the 'elbow') — that's a defensible choice for final cluster count. Pick that k, rerun KMeans once more with it, and move on to FastAPI integration.


In [17]:
# ---------------------------------------------------------------
# 14. Final clustering — k=4, chosen from the elbow analysis above
#     (biggest single drop is 2->3; the decline flattens into small,
#     roughly-equal steps from 4 onward, so 4 is the practical cutoff)
# ---------------------------------------------------------------
FINAL_K = 4
print(f"\n--- Final clustering with k={FINAL_K} ---")

kmeans_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
final_cluster_labels = kmeans_final.fit_predict(user_factors_c)

print(f"Final cluster sizes: {np.bincount(final_cluster_labels)}")

for cluster_id in range(FINAL_K):
    users_in_c = [idx_to_user_c[i] for i in range(len(final_cluster_labels)) if final_cluster_labels[i] == cluster_id]
    if not users_in_c:
        continue
    sample_u = users_in_c[0]
    top_movies = ratings_clean[(ratings_clean["userId"] == sample_u) & (ratings_clean["rating"] >= 4.0)] \
        .merge(movies_clean, on="movieId_tmdb", how="left")
    print(f"\nCluster {cluster_id} ({len(users_in_c)} users) — sample user {sample_u}'s top movies:")
    print(top_movies[["title", "rating"]].head(6).to_string(index=False))


--- Final clustering with k=4 ---
Final cluster sizes: [17380  1071  4654  2767]

Cluster 0 (17380 users) — sample user 1's top movies:
                 title  rating
The Basketball Diaries     4.5
         The Godfather     5.0
The Godfather: Part II     5.0
    Dead Poets Society     5.0
    The Breakfast Club     4.0
       The Sixth Sense     4.5

Cluster 1 (1071 users) — sample user 49's top movies:
             title  rating
         Toy Story     5.0
           Jumanji     4.5
         GoldenEye     5.0
    Twelve Monkeys     4.0
             Se7en     5.0
The Usual Suspects     5.0

Cluster 2 (4654 users) — sample user 11's top movies:
                   title  rating
                    Babe     4.0
            Pulp Fiction     4.0
The Shawshank Redemption     4.5
            Forrest Gump     4.0
          Mrs. Doubtfire     4.0
        Schindler's List     4.5

Cluster 3 (2767 users) — sample user 67's top movies:
                     title  rating
                 Toy Story